In [1]:
import re
from collections import defaultdict
from typing import List
from utils.exploit_gates1 import NetlistParser

In [2]:
import re
from collections import defaultdict
from typing import List
from detection_codes.does_have_trojan3 import parse_verilog_signals, signals_to_dict
from utils.exploit_gates2 import NetlistParser
from utils.exploit_gates2 import transform_and_parse_with_originals







# Extract Trojan gates from the reference file
def extract_trojan_gates(filename):
    trojan_gates = []
    inside_block = False

    with open(filename, 'r') as file:
        for line in file:
            stripped = line.strip()
            if stripped == "TROJAN_GATES":
                inside_block = True
                continue
            if stripped == "END_TROJAN_GATES":
                break
            if inside_block:
                trojan_gates.append(stripped)

    return trojan_gates




class Signal:
    def __init__(self, name: str, size: int, sig_type: str):
        self.name = name
        self.size = size
        self.type = sig_type  # "None", "PI", or "PO"

    def __repr__(self):
        return f"Signal(name='{self.name}', size={self.size}, type='{self.type}')"


def strip_comments(text: str) -> str:
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)
    text = re.sub(r"//.*?$", "", text, flags=re.M)
    return text


def parse_range_to_size(range_str: str) -> int:
    m = re.match(r"\[\s*(\d+)\s*:\s*(\d+)\s*\]", range_str) if range_str else None
    if not m:
        return 1
    a, b = int(m.group(1)), int(m.group(2))
    return abs(a - b) + 1


def extract_decl_names(decl_body: str):
    parts = [p.strip() for p in decl_body.split(",")]
    names = []
    for p in parts:
        base = re.split(r"\s*\[", p)[0].strip()
        base = re.split(r"\s*=", base)[0].strip()
        if base:
            names.append(base)
    return names


def parse_verilog_signals(text: str):
    clean = strip_comments(text)
    clean = re.sub(r"\s+", " ", clean)

    port_dir_map = {}
    for dir_kw in ["input", "output"]:
        pattern = rf"\b{dir_kw}\b\s+(?:reg\s+|wire\s+)?(?P<range>\[[^\]]+\]\s+)?(?P<names>[^;]+?)\s*;"
        for m in re.finditer(pattern, clean):
            rng = m.group("range")
            size = parse_range_to_size(rng) if rng else 1
            names = extract_decl_names(m.group("names"))
            for nm in names:
                if nm not in port_dir_map:
                    port_dir_map[nm] = (dir_kw, size)

    wire_info = {}
    for m in re.finditer(r"\bwire\b\s+(?P<range>\[[^\]]+\]\s+)?(?P<names>[^;]+?)\s*;", clean):
        rng = m.group("range")
        size = parse_range_to_size(rng) if rng else 1
        names = extract_decl_names(m.group("names"))
        for nm in names:
            prev = wire_info.get(nm)
            if prev is None or size > prev:
                wire_info[nm] = size

    all_names = set(wire_info.keys()) | set(port_dir_map.keys())
    signals = []
    for nm in sorted(all_names):
        if nm in port_dir_map:
            dir_kw, port_size = port_dir_map[nm]
            sig_type = "PI" if dir_kw == "input" else "PO"
            size = max(port_size, wire_info.get(nm, port_size))
        else:
            sig_type = "None"
            size = wire_info.get(nm, 1)
        signals.append(Signal(nm, size, sig_type))

    return signals

def signals_to_dict(signals):
    """
    Convert a list of Signal objects into a dictionary mapping
    signal name -> Signal object.
    """
    return {sig.name: sig for sig in signals}







def does_have_trojan9(target_file: str) -> List:
    trojan_gates = []
    trojan_gates_set = set()

    new_parser = NetlistParser()
    new_parser.parse_netlist(target_file)

    with open(target_file, "r") as f:
        text = f.read()
    signals = parse_verilog_signals(text)
    signal_dict = signals_to_dict(signals)

    def find_fanin_cone(gate_name: str) -> set:
        """
        Find the fanin cone of a gate until primary inputs or a dff using bfs.
        """
        gate = new_parser.get_gate(gate_name)
        fanin_inputs = set()
        queue = [gate]
        visited = set()
        visited.add(gate.name)
        while queue:
            current_gate = queue.pop(0)
            inputs = current_gate.inputs
            if current_gate.gate_type == 'dff':
                fanin_inputs.add(current_gate.output_net.split('[')[0])
                continue
            for input_name in inputs:
                driver_gate = new_parser.get_gate(input_name)
                if driver_gate is None:
                    # print (input_name)
                    if input_name[0] != '1':
                        if signal_dict[input_name.split('[')[0]].size > 1 and signal_dict[input_name.split('[')[0]].size <= 3:
                            fanin_inputs.add(input_name)
                        else:
                            fanin_inputs.add(input_name.split('[')[0])

                else:
                    if driver_gate.name not in visited:
                        queue.append(driver_gate)
                        visited.add(driver_gate.name)
        return fanin_inputs

    def find_whole_fanin_cone(gate_name: str) -> set:
        """
        Find the fanin cone of a gate until primary inputs or a dff using bfs.
        """
        gate = new_parser.get_gate(gate_name)
        fanin_inputs = set()
        queue = [gate]
        visited = set()
        visited.add(gate.name)
        while queue:
            current_gate = queue.pop(0)
            inputs = current_gate.inputs
            if current_gate.gate_type == 'dff':
                fanin_inputs.add(current_gate.output_net.split('[')[0])
                continue
            for input_name in inputs:
                driver_gate = new_parser.get_gate(input_name)
                if driver_gate is None:
                    # print (input_name)
                    if input_name[0] != '1':
                        if signal_dict[input_name.split('[')[0]].size > 1 and signal_dict[input_name.split('[')[0]].size <= 3:
                            fanin_inputs.add(input_name)
                        else:
                            fanin_inputs.add(input_name.split('[')[0])

                else:
                    if driver_gate.name not in visited:
                        queue.append(driver_gate)
                        visited.add(driver_gate.name)
        return visited

    ans = 0
    eligible_outputs = []
    for gate_name in new_parser.gates:
        gate = new_parser.get_gate(gate_name)
        gate_outputs = gate.outputs
        # flag = False
        # for output in gate_outputs:
        #     if output.gate_type != 'dff':
        #         flag = True
        #         break
        # if flag:
        #     continue
        if gate.gate_type != 'dff':
            #print(gate_name)
            brackets = set()
            num_of_big = 0
            fanin_inputs = find_fanin_cone(gate_name)
            if len(fanin_inputs) >= 5 and len(fanin_inputs) <= 25:
                for net in fanin_inputs:
                    if '[' in net:
                        brackets.add(net.split('[')[0])
                if len(brackets) > 2:
                    continue
                # for net in fanin_inputs:
                #     if '[' not in net:
                #         if signal_dict[net].size > 2:
                #             num_of_big += 1
                # if num_of_big < 3:
                #     continue
                if '[' not in gate.output_net:
                    continue
                if signal_dict[gate.output_net.split('[')[0]].size > 64:
                    continue
                eligible_outputs.append(gate_name)
                # print(f"Gate: {gate_name}, Fanin size: {len(fanin_inputs)}, Fanin: {sorted(fanin_inputs)}, Output: {gate.output_net}")
                ans += 1

    if ans < 4:
        return [[], False]
    
    eligible_gates = set()
    for gate_name in eligible_outputs:
        add_set = find_whole_fanin_cone(gate_name)
        eligible_gates.update(add_set)

    real_eligible_gates = set()
    for gate_name in eligible_gates:
        if '[' in gate_name:
            real_eligible_gates.add(gate_name.split('[')[0])
        else:
            real_eligible_gates.add(gate_name)

    does_exist = True
    if len(real_eligible_gates) == 0:
        does_exist = False
    trojan_gates = list(real_eligible_gates)


    return [trojan_gates, does_exist]

In [13]:
Test_Design_Number = 34
target_file = f"./release_hidden_0923/release_hidden/design{Test_Design_Number}.v"

trojan_gates = []
trojan_gates_set = set()

new_parser = NetlistParser()
new_parser.parse_netlist(target_file)

with open(target_file, "r") as f:
    text = f.read()
signals = parse_verilog_signals(text)
signal_dict = signals_to_dict(signals)

def find_fanin_cone(gate_name: str) -> set:
    """
    Find the fanin cone of a gate until primary inputs or a dff using bfs.
    """
    gate = new_parser.get_gate(gate_name)
    fanin_inputs = set()
    queue = [gate]
    visited = set()
    visited.add(gate.name)
    while queue:
        current_gate = queue.pop(0)
        inputs = current_gate.inputs
        if current_gate.gate_type == 'dff':
            fanin_inputs.add(current_gate.output_net.split('[')[0])
            continue
        for input_name in inputs:
            driver_gate = new_parser.get_gate(input_name)
            if driver_gate is None:
                # print (input_name)
                if input_name[0] != '1':
                    if signal_dict[input_name.split('[')[0]].size > 1 and signal_dict[input_name.split('[')[0]].size <= 3:
                        fanin_inputs.add(input_name)
                    else:
                        fanin_inputs.add(input_name.split('[')[0])

            else:
                if driver_gate.name not in visited:
                    queue.append(driver_gate)
                    visited.add(driver_gate.name)
    return fanin_inputs

In [9]:
def find_whole_fanin_cone(gate_name: str) -> set:
    """
    Find the fanin cone of a gate until primary inputs or a dff using bfs.
    """
    gate = new_parser.get_gate(gate_name)
    fanin_inputs = set()
    queue = [gate]
    visited = set()
    visited.add(gate.name)
    while queue:
        current_gate = queue.pop(0)
        inputs = current_gate.inputs
        if current_gate.gate_type == 'dff':
            fanin_inputs.add(current_gate.output_net.split('[')[0])
            continue
        for input_name in inputs:
            driver_gate = new_parser.get_gate(input_name)
            if driver_gate is None:
                # print (input_name)
                if input_name[0] != '1':
                    if signal_dict[input_name.split('[')[0]].size > 1 and signal_dict[input_name.split('[')[0]].size <= 3:
                        fanin_inputs.add(input_name)
                    else:
                        fanin_inputs.add(input_name.split('[')[0])

            else:
                if driver_gate.name not in visited:
                    queue.append(driver_gate)
                    visited.add(driver_gate.name)
    return visited

In [10]:
ans = 0
eligible_outputs = []
for gate_name in new_parser.gates:
    gate = new_parser.get_gate(gate_name)
    gate_outputs = gate.outputs
    # flag = False
    # for output in gate_outputs:
    #     if output.gate_type != 'dff':
    #         flag = True
    #         break
    # if flag:
    #     continue
    if gate.gate_type != 'dff':
        #print(gate_name)
        brackets = set()
        num_of_big = 0
        fanin_inputs = find_fanin_cone(gate_name)
        if len(fanin_inputs) >= 5 and len(fanin_inputs) <= 25:
            for net in fanin_inputs:
                if '[' in net:
                    brackets.add(net.split('[')[0])
            if len(brackets) > 2:
                continue
            # for net in fanin_inputs:
            #     if '[' not in net:
            #         if signal_dict[net].size > 2:
            #             num_of_big += 1
            # if num_of_big < 3:
            #     continue
            if '[' not in gate.output_net:
                continue
            if signal_dict[gate.output_net.split('[')[0]].size > 64:
                continue
            eligible_outputs.append(gate_name)
            # print(f"Gate: {gate_name}, Fanin size: {len(fanin_inputs)}, Fanin: {sorted(fanin_inputs)}, Output: {gate.output_net}")
            ans += 1

if ans < 4:
    print([[], False])

In [11]:
eligible_gates = set()
for gate_name in eligible_outputs:
    add_set = find_whole_fanin_cone(gate_name)
    eligible_gates.update(add_set)

real_eligible_gates = set()
for gate_name in eligible_gates:
    if '[' in gate_name:
        real_eligible_gates.add(gate_name.split('[')[0])
    else:
        real_eligible_gates.add(gate_name)

does_exist = True
if len(real_eligible_gates) == 0:
    does_exist = False
trojan_gates = list(real_eligible_gates)


print([trojan_gates, does_exist])

[['g24', 'g2', 'g34', 'g22', 'g11', 'g39', 'g36', 'g44', 'g26', 'g71', 'g7', 'g25', 'g17', 'g3', 'g69', 'g57', 'g16', 'g42', 'g15', 'g46', 'g55', 'g45', 'g28', 'g0', 'g67', 'g12', 'g9', 'g33', 'g18', 'g72', 'g41', 'g63', 'g32', 'g40', 'g8', 'g60', 'g38', 'g61', 'g35', 'g6', 'g4', 'g21', 'g37', 'g14', 'g13', 'g66'], True]


In [12]:


parser = NetlistParser()
parser.parse_netlist(target_file)
reference_Trojans_file = f"./release_hidden_0923/release_hidden/result{Test_Design_Number}.txt"
actual_trojan_gates = extract_trojan_gates(reference_Trojans_file)
if not does_exist:
    score = 0
    print_excel_outputs = [Test_Design_Number, 0, 0, len(actual_trojan_gates), len(parser.gates) - len(actual_trojan_gates), 0, 0, 0, score]
    print('\t'.join(map(str, print_excel_outputs)))
else:
    score = 2
# print(f"Number of trojan gates in reference file: {len(actual_trojan_gates)}")

# print (f"Actual Trojan gates: {sorted(actual_trojan_gates, key=lambda x: (len(x), x), reverse=True)}")
# print (f"Predicted Trojan gates: {sorted(trojan_gates, key=lambda x: (len(x), x), reverse=True)}")


true_positive = 0
for gate in parser.gates:
    if gate in trojan_gates and gate in actual_trojan_gates:
        true_positive = true_positive + 1


#false_positive = len(trojan_gates - set(actual_trojan_gates))
false_positive = 0
for gate in trojan_gates:
    if gate not in actual_trojan_gates:
        false_positive = false_positive + 1

#false_negative = len(set(actual_trojan_gates) - trojan_gates)
false_negative = 0
for gate in actual_trojan_gates:
    if gate not in trojan_gates:
        false_negative = false_negative + 1

true_negative = 0
for gate in parser.gates:
    if gate not in trojan_gates and gate not in actual_trojan_gates:
        true_negative = true_negative + 1


TPR = true_positive / len(actual_trojan_gates) if actual_trojan_gates else 0
FPR = false_positive / (len(actual_trojan_gates) + false_negative) if (len(actual_trojan_gates) + false_negative) > 0 else 0
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
f1_score = 2 * (recall * precision) / (recall + precision) if (recall + precision) > 0 else 0

score = score + f1_score

# print(f"True Positive Rate (TPR): {TPR:.4f}")
# print(f"False Positive Rate (FPR): {FPR:.4f}")
# print(f"True Positive Count: {true_positive}")
# print(f"False Positive Count: {false_positive}")
# print(f"False Negative Count: {false_negative}")
# print(f"True Negative Count: {true_negative}")
# print (f"precision: {precision:.4f}")
# print (f"recall: {TPR:.4f}")
# print (f"F1 score: {f1_score:.4f}")
# print (f"total number of gates: {len(parser.gates)}")
# print_excel_outputs = [true_positive, false_positive, false_negative, true_negative, precision, TPR, f1_score]
# print('\t'.join(map(str, print_excel_outputs)))
# print(Test_Design_Number)

# for gate_name in actual_trojan_gates:
#     if gate_name not in trojan_gates:
#         print(parser.get_gate(gate_name))
print_excel_outputs = [Test_Design_Number,true_positive, false_positive, false_negative, true_negative, precision, TPR, f1_score, score]
print('\t'.join(map(str, print_excel_outputs)))

28	38	8	0	28	0.8260869565217391	1.0	0.9047619047619047	2.9047619047619047
